# 03 BehaviorVariable Inference

## 1. Data

In [1]:
from pathlib import Path
import numpy as np
import pandas as pd
from scipy import stats
from IPython.display import display, Markdown

ROOT = Path.cwd().resolve().parent
PROCESSED_PATH = ROOT / "data" / "processed" / "yrbs_combined_processed.csv"
TAB_DIR = ROOT / "outputs" / "tables"
SUM_DIR = ROOT / "outputs" / "summary"

BEHAVIOR_VAR = "SadOrHopeless"
BEHAVIOR_P0 = 0.30
CONT_VAR = "HowMuchDoYouWeighWithoutShoesInKG"
CONT_MU0 = 68.0

processed = pd.read_csv(PROCESSED_PATH)
print("Processed data shape:", processed.shape)

Processed data shape: (14041, 10)


## 2.Proportion Inference: `SadOrHopeless`

In [2]:
sad_binary = processed["sad_binary"].dropna().astype(int)
n_b = int(sad_binary.shape[0])
x_b = int((sad_binary == 1).sum())
phat = x_b / n_b

ci_b = stats.binomtest(x_b, n_b).proportion_ci(confidence_level=0.95, method="wilson")
z_stat = (phat - BEHAVIOR_P0) / np.sqrt(BEHAVIOR_P0 * (1 - BEHAVIOR_P0) / n_b)
p_value_b = 2 * (1 - stats.norm.cdf(abs(z_stat)))
p_value_display = "< 0.0001" if p_value_b < 0.0001 else f"{p_value_b:.4f}"

prop_results = pd.DataFrame({
    "quantity": [
        "sample_size", "successes", "sample_proportion",
        "benchmark_p0", "z_statistic", "p_value", "ci_low", "ci_high"
    ],
    "value": [
        str(n_b),
        str(x_b),
        f"{phat:.4f}",
        f"{BEHAVIOR_P0:.4f}",
        f"{z_stat:.4f}",
        p_value_display,
        f"{ci_b.low:.4f}",
        f"{ci_b.high:.4f}"
    ]
})

decision_b = "reject H0" if p_value_b < 0.05 else "fail to reject H0"

display(prop_results)

# Save
behavior_summary = pd.DataFrame({
    "analysis": ["Population proportion"],
    "variable": [BEHAVIOR_VAR],
    "estimate": [f"{phat:.4f}"],
    "benchmark": [f"{BEHAVIOR_P0:.4f}"],
    "test_statistic": [f"{z_stat:.4f}"],
    "p_value": [
        "< 0.0001" if p_value_b < 0.0001 else f"{p_value_b:.4f}"
    ],
    "ci_low": [f"{ci_b.low:.4f}"],
    "ci_high": [f"{ci_b.high:.4f}"],
    "decision_5pct": [decision_b]
})

behavior_summary.to_csv(TAB_DIR / "03_behavior_inference_summary.csv", index=False)

,quantity,value
0,sample_size,13845
1,successes,4153
2,sample_proportion,0.3000
3,benchmark_p0,0.3000
4,z_statistic,-0.0093
5,p_value,0.9926
6,ci_low,0.2924
7,ci_high,0.3077


### Interpretation (Proportion Analysis)

- We used the sample to estimate and test whether the population proportion for **SadOrHopeless** differs from the benchmark of **0.30**.
- The sample proportion is **0.3000**, which is almost identical to the benchmark proportion of **0.30**.
- The z-statistic is very close to **0**, so the observed difference is too small to suggest a meaningful departure from the null hypothesis.
- The **95% confidence interval** is approximately **(0.2924, 0.3077)**, which contains **0.30**.
- Because the **p-value is 0.9926**, we **fail to reject the null hypothesis** at the 5% significance level.
- In context, the sample does not provide enough evidence to conclude that the population proportion of **SadOrHopeless** is different from **0.30**.
- This is also consistent with the EDA, where the Success (1) group was clearly less common than the Failure (0) group, and the Success proportion appeared visually close to **0.30**.
- A cautious point is that this result does not prove the true population proportion is exactly 0.30; it only means the sample does not provide enough evidence that it is different from 0.30.